In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
project_path = "/content/drive/MyDrive/Data_Cleaning_Project"
print(os.path.exists(project_path))# False initially because the folder was nor created

folders = [
    "01_Raw_Data",
    "02_Cleaned_Data",
    "03_Reports",
    "04_Documentation",
    "05_Notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)
print("Project folders created successfully!")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
Project folders created successfully!


In [ ]:
for root, dirs, files in os.walk(project_path):
    print(root)

/content/drive/MyDrive/Data_Cleaning_Project
/content/drive/MyDrive/Data_Cleaning_Project/01_Raw_Data
/content/drive/MyDrive/Data_Cleaning_Project/02_Cleaned_Data
/content/drive/MyDrive/Data_Cleaning_Project/03_Reports
/content/drive/MyDrive/Data_Cleaning_Project/04_Documentation
/content/drive/MyDrive/Data_Cleaning_Project/05_Notebooks


In [2]:
import pandas as pd
import numpy as np

print(pd.__version__) # pandas version 2.2.3
print("Hello world") # This is just for the testing purpose


2.2.3
Hello world


# 1.Data Inspection

In [63]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Data_Cleaning_Project/01_Raw_Data/linkup-churn.csv")

df.head()
df.shape # 4000 rows and 12 columns total before cleaninig
df.info() #
df.describe()
df["churn"].value_counts(dropna=False) #This returns the all the inconsistant items which is enterd in the data set
df.columns
df.tail()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        4000 non-null   object 
 1   senior_citizen     4000 non-null   object 
 2   tenure_months      4000 non-null   int64  
 3   contract           4000 non-null   object 
 4   internet_service   3179 non-null   object 
 5   streaming_tv       4000 non-null   object 
 6   paperless_billing  4000 non-null   object 
 7   payment_method     3880 non-null   object 
 8   monthly_charges    4000 non-null   float64
 9   total_charges      4000 non-null   object 
 10  support_calls      4000 non-null   int64  
 11  churn              4000 non-null   object 
dtypes: float64(1), int64(2), object(9)
memory usage: 375.1+ KB


,customer_id,senior_citizen,tenure_months,contract,internet_service,streaming_tv,paperless_billing,payment_method,monthly_charges,total_charges,support_calls,churn
3995,LU-3762601,No,24,Two year,NaN,No internet,Yes,Mailed check,25.07,590.08,1,no
3996,LU-5388626,No,47,Two year,DSL,Yes,Yes,Mailed check,53.58,2492.0,0,No
3997,LU-5340327,No,52,Two year,DSL,No,No,NaN,51.07,2766.37,2,0
3998,LU-7985536,No,15,One year,DSL,Yes,No,Mailed check,38.00,545.58,2,Yes
3999,LU-1925574,No,67,One year,Fiber,No,No,Electronic check,67.10,4368.32,2,No


# 2.Data Cleaning handeling Missing values

In [76]:
cdf = df.copy() # made a copy of the data
cdf.isnull().sum() # Identified internet service column (821) and payment method column (120) have missing values

cdf["internet_service"] = cdf["internet_service"].fillna("Unknown")
cdf["internet_service"].head(10)

cdf["payment_method"].head(20)
cdf["payment_method"]=cdf["payment_method"].fillna("Unknown")
cdf["payment_method"].head(20)

cdf.isnull().sum() # There are no empty values in the data set



,0
customer_id,0
senior_citizen,0
tenure_months,0
contract,0
internet_service,0
streaming_tv,0
paperless_billing,0
payment_method,0
monthly_charges,0
total_charges,0


# 3.Changing Datatype and repalceing the invalid values

In [139]:
# First, restore the original uncleaned churn column from our backup copy df
cdf["churn"] = df["churn"]

# Map the values using a dictionary (handling common variations with lowercasing and stripping whitespace for safety)
cdf["churn"] = (
    cdf["churn"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes": True,
        "y": True,
        "1": True,
        "true": True,
        "no": False,
        "n": False,
        "0": False,
        "false": False
    })
)

# Let's check the result now
display(cdf["churn"].value_counts(dropna=False))
cdf["churn"].dtype


cdf["paperless_billing"] = df["paperless_billing"]
cdf["paperless_billing"] = (
    cdf["paperless_billing"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes":True,
        "no":False
    })
)
display(cdf["paperless_billing"].value_counts(dropna=False))
cdf["paperless_billing"].dtypes


display(cdf["streaming_tv"].value_counts(dropna=False)) # Converted the no intenet values to the False as it is assumed to be False
cdf["streaming_tv"].dtypes
cdf["streaming_tv"]=cdf["streaming_tv"].fillna("No internet")



cdf["senior_citizen"].dtypes
cdf["senior_citizen"] = df["senior_citizen"]
cdf["senior_citizen"] = (
    cdf["senior_citizen"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "yes":True,
        "no":False
    }
    )
)
display(cdf["senior_citizen"].value_counts(dropna=False))
cdf["senior_citizen"].dtypes


,count
churn,
False,2812
True,1188


,count
paperless_billing,
True,2410
False,1590


,count
streaming_tv,
True,1597
False,1582
No internet,821


,count
senior_citizen,
False,3366
True,634


dtype('bool')

In [140]:

cdf.head(30)

,customer_id,senior_citizen,tenure_months,contract,internet_service,streaming_tv,paperless_billing,payment_method,monthly_charges,total_charges,support_calls,churn
0,LU-8416758,False,50,Month-to-month,Fiber,False,True,Bank transfer,68.13,3406.74,1,False
1,LU-7459452,True,6,One year,DSL,True,True,Credit card,43.78,265.87,3,True
2,LU-0871639,False,49,One year,Fiber,True,False,Electronic check,72.26,3434.11,3,True
3,LU-9990453,True,39,One year,DSL,False,True,Mailed check,67.90,2613.14,0,False
4,LU-3394999,False,1,Month-to-month,Unknown,No internet,False,Credit card,21.10,20.4,0,True
5,LU-3997187,False,66,Month-to-month,Fiber,True,False,Credit card,70.68,4547.6,0,True
6,LU-3536502,False,8,One year,DSL,False,True,Unknown,50.51,392.34,2,False
7,LU-7140153,False,71,One year,Fiber,False,True,Mailed check,73.64,5090.16,1,False
8,LU-4123168,False,61,One year,Fiber,False,True,Mailed check,67.72,3993.15,1,False
9,LU-0216265,False,61,Month-to-month,DSL,True,False,Bank transfer,53.42,3135.57,1,False


In [143]:
from google.colab import drive
import os
drive.mount('/content/drive')

cdf.to_csv("/content/drive/MyDrive/Data_Cleaning_Project/02_Cleaned_Data/Cleaned_data_of_link_up_churn.csv",index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [145]:

import os
import base64
import requests
from google.colab import userdata

# 1. Notebook location in Google Drive
notebook_path = "/content/drive/MyDrive/Data_Cleaning_Project/05_Notebooks/data_Cleaning_project.ipynb"


# 2. GitHub repository details
github_username = "faizankhan291106"
repository_name = "telecom-customer-data-cleaning"
github_file_path = "telecom_data_cleaning.ipynb"

# 3. Get GitHub token securely from Colab Secrets
token = userdata.get("GITHUB_TOKEN")

# 4. Read the notebook
with open(notebook_path, "rb") as file:
    notebook_content = base64.b64encode(file.read()).decode("utf-8")

# 5. GitHub API configuration
api_url = f"https://api.github.com/repos/{github_username}/{repository_name}/contents/{github_file_path}"

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}

# 6. Check whether the notebook already exists
response = requests.get(api_url, headers=headers)

data = {
    "message": "Fully cleaned the dataset a",
    "content": notebook_content
}

if response.status_code == 200:
    data["sha"] = response.json()["sha"]
elif response.status_code != 404:
    raise Exception(f"GitHub error: {response.status_code} - {response.text}")

# 7. Upload or update the notebook
upload_response = requests.put(api_url, headers=headers, json=data)

if upload_response.status_code in [200, 201]:
    print("Success! Notebook uploaded/updated in GitHub.")
    print("Commit:", upload_response.json()["commit"]["message"])
else:
    print("Upload failed:", upload_response.status_code)
    print(upload_response.text)


Success! Notebook uploaded/updated in GitHub.
Commit: Update data cleaning notebook
